# JOILang Cloud-only-by-config Advisor Manual Intervention Lab

목적: `cloud-only-by-config` 방식으로 GA를 먼저 1~5세대 실행한 뒤, 사용자가 직접 `advisor_feedback.py`를 수정하고 같은 `ga_output`에 generation 6, 7, ...을 한 세대씩 resume하여 다음을 확인한다.

- generation별 `advisor_prompt_generation_XXX.txt` 위치와 내용
- cloud advisor prompt 수정 전후 차이
- advisor response/proposal/accept/reject 변화
- JOILang code generation prompt/block 구성의 before-after
- DETPass, token, transition, compression/fallback leakage 변화
- 실험 개입 이력과 산출물 저장

전제: 현재 repo는 strict cloud-only flag가 아니라 **quota/rate 설정 기반**으로 cloud-only처럼 실행한다.

```text
cloud-only-by-config =
  use_advisor=True
  advisor_compression_child_quota > 0
  local compression quota/rate = 0
  artifact에서 source="compression_fallback" == 0인지 확인
```


In [9]:
# ============================================================
# Cell 1. Server/path configuration
# A6000 default. Change SERVER_PRESET only when using A100.
# IMPORTANT: do not resolve conda/venv Python symlink.
# ============================================================

from __future__ import annotations

from pathlib import Path
from datetime import datetime
import os
import sys
import re
import json
import time
import math
import ast
import shlex
import shutil
import subprocess
import traceback
import difflib
import hashlib

import pandas as pd
import numpy as np

SERVER_PRESET = os.environ.get("JOILANG_SERVER_PRESET", "A100_SET_B")  # "A6000_SET_A" or "A100_SET_B"

RUN_INITIAL_SMOKE = True           # Cell 6: run generation 1..INITIAL_GENS
RUN_RESUME_ONE_GEN = False         # Cell 10: set True only after editing advisor_feedback.py
RUN_FULL_CLOUD_ONLY = False
RUN_FULL_THREE_WAY = False

INITIAL_GENS = 2                   # initial run ends at generation 5
NEXT_GEN = None                    # None => infer max_generation + 1 from progress.csv

SMOKE_MODEL_KEY = os.environ.get("JOILANG_SMOKE_MODEL_KEY", "qwen25_coder_14b")
SMOKE_CATEGORIES = (3, 4, 5, 6)
SMOKE_LIMIT_PER_CATEGORY = 1
SMOKE_SAMPLE_SIZE = 4
SMOKE_VALIDATION_SIZE = 4
SMOKE_POPULATION = 5

ADVISOR_MODEL_KEY = os.environ.get("JOILANG_ADVISOR_MODEL_KEY", "gpt41_mini")

SERVER_CONFIGS = {
    "A100_SET_B": {
        "repo": Path("/root/llm/JOILang-Server"),
        "python": Path("/root/llm/je/bin/python"),  # do not resolve
        "local_model_base": Path("/root/llm/local_models"),
    },
    "A6000_SET_A": {
        "repo": Path("/home/mgjeong/Desktop/llm/JOILang-Server"),
        "python": Path("/home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10"),  # do not resolve
        "local_model_base": Path("/home/mgjeong/Desktop/llm/local_models"),
    },
}

if SERVER_PRESET not in SERVER_CONFIGS:
    raise ValueError(f"Unknown SERVER_PRESET={SERVER_PRESET}")

cfg = SERVER_CONFIGS[SERVER_PRESET]
REPO = cfg["repo"]
PYTHON = cfg["python"]
LOCAL_MODEL_BASE = cfg["local_model_base"]

VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPTS_DIR = VERSION_DIR / "scripts"
TESTS_DIR = VERSION_DIR / "tests"
SCRIPT = SCRIPTS_DIR / "run_ga_search.py"
ADVISOR_FEEDBACK_PATH = SCRIPTS_DIR / "advisor_feedback.py"

RESULTS_ROOT = VERSION_DIR / "results"
NOTEBOOK_DIR = VERSION_DIR / "notebooks"
PAPER_ARTIFACT_ROOT = RESULTS_ROOT / "paper_artifacts"
SUMMARY_DIR = PAPER_ARTIFACT_ROOT / "cloud_only_by_config_manual"
FIGURE_DIR = SUMMARY_DIR / "figures"
TABLE_DIR = SUMMARY_DIR / "tables"
PATCH_DIR = SUMMARY_DIR / "advisor_feedback_snapshots"

for d in [RESULTS_ROOT, NOTEBOOK_DIR, PAPER_ARTIFACT_ROOT, SUMMARY_DIR, FIGURE_DIR, TABLE_DIR, PATCH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
    "phi35_mini": "phi35_mini",
    "gemma2_9b_it": "gemma2_9b_it",
}

MODEL_LIST_3 = [
    ("7B", "qwen25_coder_7b"),
    ("8B", "llama31_8b"),
    ("14B", "qwen25_coder_14b"),
]

# Offline/local model environment.
os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"
os.environ["TRANSFORMERS_OFFLINE"] = os.environ.get("TRANSFORMERS_OFFLINE", "1")
os.environ["HF_HUB_OFFLINE"] = os.environ.get("HF_HUB_OFFLINE", "1")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("=" * 120)
print("[CONFIG]")
print("=" * 120)
print("SERVER_PRESET:", SERVER_PRESET)
for k, v in {
    "REPO": REPO,
    "PYTHON": PYTHON,
    "LOCAL_MODEL_BASE": LOCAL_MODEL_BASE,
    "VERSION_DIR": VERSION_DIR,
    "SCRIPT": SCRIPT,
    "ADVISOR_FEEDBACK_PATH": ADVISOR_FEEDBACK_PATH,
    "RESULTS_ROOT": RESULTS_ROOT,
    "SUMMARY_DIR": SUMMARY_DIR,
    "PATCH_DIR": PATCH_DIR,
}.items():
    print(f"{k:24s}: {v} exists={Path(v).exists()}")

assert REPO.exists(), REPO
assert PYTHON.exists(), PYTHON
assert SCRIPT.exists(), SCRIPT
assert ADVISOR_FEEDBACK_PATH.exists(), ADVISOR_FEEDBACK_PATH
assert LOCAL_MODEL_BASE.exists(), LOCAL_MODEL_BASE


[CONFIG]
SERVER_PRESET: A100_SET_B
REPO                    : /root/llm/JOILang-Server exists=True
PYTHON                  : /root/llm/je/bin/python exists=True
LOCAL_MODEL_BASE        : /root/llm/local_models exists=True
VERSION_DIR             : /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413 exists=True
SCRIPT                  : /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py exists=True
ADVISOR_FEEDBACK_PATH   : /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/advisor_feedback.py exists=True
RESULTS_ROOT            : /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results exists=True
SUMMARY_DIR             : /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/paper_artifacts/cloud_only_by_config_manual exists=True
PATCH_DIR               : /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/paper_artifacts/cloud_only_by_config_manual/advisor_feedback_snapshots exists=True


In [10]:
# ============================================================
# Cell 2. Environment verification
# ============================================================

def run_cmd(cmd, timeout=180, check=False, env=None):
    print("\nRUN:", " ".join(map(str, cmd)))
    p = subprocess.run(
        list(map(str, cmd)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=timeout,
        check=False,
        env=env,
    )
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed rc={p.returncode}")
    return p

env_code = """
import sys
print("python:", sys.executable)
try:
    import torch
    print("torch:", getattr(torch, "__version__", "NO_VERSION_ATTR"))
    print("cuda:", torch.cuda.is_available())
    print("device_count:", torch.cuda.device_count())
    if torch.cuda.is_available() and torch.cuda.device_count() > 0:
        print("device0:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch_error:", repr(e))
try:
    import transformers
    print("transformers:", getattr(transformers, "__version__", "NO_VERSION_ATTR"))
except Exception as e:
    print("transformers_error:", repr(e))
"""

print("=" * 120)
print("[PYTHON ENV]")
print("=" * 120)
run_cmd([PYTHON, "-c", env_code], timeout=180, check=True)

print("=" * 120)
print("[NVIDIA-SMI]")
print("=" * 120)
run_cmd(["nvidia-smi"], timeout=60, check=False)

print("=" * 120)
print("[OPENAI API KEY]")
print("=" * 120)
print("OPENAI_API_KEY exists:", bool(os.environ.get("OPENAI_API_KEY", "").strip()))

print("=" * 120)
print("[MODEL PATH CHECK]")
print("=" * 120)
rows = []
for model_key, dirname in MODEL_DIRS.items():
    p = LOCAL_MODEL_BASE / dirname
    rows.append({
        "model_key": model_key,
        "path": str(p),
        "exists": p.exists(),
        "config": (p / "config.json").exists(),
        "tokenizer": (p / "tokenizer.json").exists(),
        "index": (p / "model.safetensors.index.json").exists(),
        "safetensors_count": len(list(p.glob("*.safetensors"))) if p.exists() else 0,
    })
display(pd.DataFrame(rows))


[PYTHON ENV]

RUN: /root/llm/je/bin/python -c 
import sys
print("python:", sys.executable)
try:
    import torch
    print("torch:", getattr(torch, "__version__", "NO_VERSION_ATTR"))
    print("cuda:", torch.cuda.is_available())
    print("device_count:", torch.cuda.device_count())
    if torch.cuda.is_available() and torch.cuda.device_count() > 0:
        print("device0:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch_error:", repr(e))
try:
    import transformers
    print("transformers:", getattr(transformers, "__version__", "NO_VERSION_ATTR"))
except Exception as e:
    print("transformers_error:", repr(e))

python: /root/llm/je/bin/python
torch: 2.7.1+cu118
cuda: True
device_count: 1
device0: NVIDIA A100 80GB PCIe
transformers: 5.8.0

[NVIDIA-SMI]

RUN: nvidia-smi
Fri Jun 12 16:15:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA

,model_key,path,exists,config,tokenizer,index,safetensors_count
0,qwen25_coder_7b,/root/llm/local_models/qwen25_coder_7b,True,True,True,True,4
1,llama31_8b,/root/llm/local_models/llama31_8b,True,True,True,True,4
2,qwen25_coder_14b,/root/llm/local_models/qwen25_coder_14b,True,True,True,True,6
3,phi35_mini,/root/llm/local_models/phi35_mini,True,True,True,True,2
4,gemma2_9b_it,/root/llm/local_models/gemma2_9b_it,True,True,True,True,4


In [11]:
# ============================================================
# Cell 3. Source/flag verification
# Config-based cloud-only advisor mode.
# Strict cloud-only flags are optional; existing quota/ratio flags are required.
# ============================================================

run_ga_src = SCRIPT.read_text(encoding="utf-8", errors="replace")
advisor_src = ADVISOR_FEEDBACK_PATH.read_text(encoding="utf-8", errors="replace")

try:
    p = subprocess.run([str(PYTHON), str(SCRIPT), "--help"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=180)
    help_text = p.stdout
    print("help rc:", p.returncode)
except Exception as e:
    help_text = ""
    print("[WARN] help failed:", repr(e))

SUPPORTED_FLAGS = set(re.findall(r"--[A-Za-z0-9][A-Za-z0-9_-]*", help_text + "\n" + run_ga_src))

required_existing_flags = [
    "--llm-mutation-advisor",
    "--advisor-model-key",
    "--advisor-trigger-mode",
    "--advisor-force-child-quota",
    "--advisor-min-population-for-child",
    "--advisor-compression-child-quota",
    "--advisor-prefer-compression-after-detpass",
    "--enable-compression-mutation",
    "--compression-detpass-threshold",
    "--aggressive-compression-after-target",
    "--compression-child-quota",
    "--compression-child-ratio",
    "--compression-token-reduction-target",
    "--compression-token-plateau-delta",
    "--allow-aggressive-compression",
    "--micro-compression-child-quota",
    "--micro-compression-child-ratio",
    "--block-compression-child-quota",
    "--block-compression-child-ratio",
    "--multi-block-compression-child-quota",
    "--multi-block-compression-child-ratio",
    "--global-budget-compression-child-quota",
    "--enable-block-token-breakdown",
    "--enable-multi-block-compression",
    "--enable-render-budget-compression",
    "--min-compression-token-delta",
    "--mutation-mode",
    "--selection-mode",
    "--fitness-mode",
    "--token-penalty-mode",
    "--stop-controller-mode",
    "--reasoning-mutation-mode",
    "--intent-hint-mode",
]

optional_strict_flags = [
    "--cloud-only-advisor-compression",
    "--disable-compression-fallback",
    "--advisor-proposal-k",
    "--advisor-min-usable-compression-proposals",
    "--advisor-repair-invalid-proposals",
    "--advisor-repair-max-attempts",
    "--advisor-require-block-proposal-after-detpass",
    "--advisor-require-token-delta",
    "--advisor-disallow-genome-only-after-detpass",
    "--advisor-cloud-only-strict-schema",
    "--advisor-before-after-report",
    "--advisor-include-dataset-feedback",
    "--advisor-feedback-topk",
    "--advisor-feedback-max-failures-per-family",
    "--advisor-include-prompt-before-after-context",
    "--advisor-include-case-abc-sections",
    "--advisor-write-debug-prompt-sections",
]

optional_strict_sections = [
    "ADVISOR_ROLE",
    "CURRENT_STATE",
    "DATASET_EVALUATION_FEEDBACK",
    "PROMPT_TOKEN_BREAKDOWN",
    "BLOCK_TOKEN_BREAKDOWN",
    "PROTECTED_BLOCKS",
    "COMPRESSION_ALLOWED_BLOCKS",
    "CASE_A_MICRO_COMPRESSION",
    "CASE_B_THRESHOLD_BLOCK_COMPRESSION",
    "CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION",
    "STRICT_JSON_RESPONSE_SCHEMA",
]

rows = []
for f in required_existing_flags:
    rows.append({"kind": "required_existing_flag", "name": f, "present": f in SUPPORTED_FLAGS, "required_for_by_config": True})
for f in optional_strict_flags:
    rows.append({"kind": "optional_strict_cloud_only_flag", "name": f, "present": f in SUPPORTED_FLAGS, "required_for_by_config": False})
for s in optional_strict_sections:
    rows.append({"kind": "optional_strict_prompt_section", "name": s, "present": s in advisor_src, "required_for_by_config": False})

df_flags = pd.DataFrame(rows)
display(df_flags)

missing_required = [r for r in rows if r["required_for_by_config"] and not r["present"]]
STRICT_CLOUD_ONLY_IMPLEMENTED = all(r["present"] for r in rows if r["kind"].startswith("optional_strict"))

print("=" * 120)
print("[SOURCE / FLAG VERIFICATION SUMMARY]")
print("=" * 120)
print("STRICT_CLOUD_ONLY_IMPLEMENTED:", STRICT_CLOUD_ONLY_IMPLEMENTED)

if missing_required:
    print("[ERROR] Missing required existing flags for cloud-only-by-config:")
    for r in missing_required:
        print("-", r["name"])
    raise AssertionError(missing_required)

print("[OK] Existing quota/ratio strong compression flags are available.")
print("[MODE] This notebook uses cloud-only-by-config, not strict cloud-only flags.")


help rc: 0


,kind,name,present,required_for_by_config
0,required_existing_flag,--llm-mutation-advisor,True,True
1,required_existing_flag,--advisor-model-key,True,True
2,required_existing_flag,--advisor-trigger-mode,True,True
3,required_existing_flag,--advisor-force-child-quota,True,True
4,required_existing_flag,--advisor-min-population-for-child,True,True
...,...,...,...,...
56,optional_strict_prompt_section,COMPRESSION_ALLOWED_BLOCKS,False,False
57,optional_strict_prompt_section,CASE_A_MICRO_COMPRESSION,False,False
58,optional_strict_prompt_section,CASE_B_THRESHOLD_BLOCK_COMPRESSION,False,False
59,optional_strict_prompt_section,CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION,False,False


[SOURCE / FLAG VERIFICATION SUMMARY]
STRICT_CLOUD_ONLY_IMPLEMENTED: False
[OK] Existing quota/ratio strong compression flags are available.
[MODE] This notebook uses cloud-only-by-config, not strict cloud-only flags.


In [12]:
# ============================================================
# Cell 4. Utility helpers
# ============================================================

def timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8", errors="replace"))
    except Exception as e:
        print("[JSON READ ERROR]", path, repr(e))
        return None

def read_jsonl(path):
    rows = []
    path = Path(path)
    if not path.exists():
        return rows
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except Exception:
            rows.append({"_raw": line, "_parse_error": True})
    return rows

def safe_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as e:
        print("[CSV READ ERROR]", path, repr(e))
        return pd.DataFrame()

def sha256_file(path):
    path = Path(path)
    if not path.exists():
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def classify_case_abc(row):
    level = str(row.get("compression_level", "") or "").lower()
    op = str(row.get("operator", row.get("mutation_type", row.get("mutation", ""))) or "").lower()
    target = str(row.get("target_block_id", row.get("selected_block_id", "")) or "").lower()
    ids = row.get("selected_block_ids", [])
    if isinstance(ids, str):
        try:
            ids = ast.literal_eval(ids)
        except Exception:
            ids = []
    if level in {"multi_block", "global_budget", "global", "render_budget"}:
        return "Case C"
    if "multi" in op or "global" in op or "budget" in op:
        return "Case C"
    if isinstance(ids, list) and len(ids) >= 2:
        return "Case C"
    if level == "block":
        return "Case B"
    if target and target not in {"genome", "none", "nan", ""}:
        return "Case B"
    if "drop_optional_block" in op or "compact_reasoning_skeleton" in op:
        return "Case B"
    if level == "micro":
        return "Case A"
    if any(k in op for k in ["candidate_strategies", "lower_output_max_tokens", "template_compress", "dedupe", "safe"]):
        return "Case A"
    if any(k in op for k in ["few_shot", "micro_rules", "compact_block_params"]):
        return "Case B"
    return "Unclassified"

def load_progress(out_dir):
    return safe_csv(Path(out_dir) / "ga_generation_progress.csv")

def load_transitions(out_dir):
    return safe_csv(Path(out_dir) / "population_transitions.csv")

def load_all_proposals(out_dir):
    out_dir = Path(out_dir)
    rows = []
    for fn in ["advisor_mutation_proposals.jsonl", "mutation_proposals.jsonl"]:
        for r in read_jsonl(out_dir / fn):
            rr = dict(r)
            rr["_file"] = fn
            if "operator" not in rr:
                rr["operator"] = rr.get("mutation_type", rr.get("mutation", ""))
            rr["case_label_inferred"] = classify_case_abc(rr)
            rows.append(rr)
    return pd.DataFrame(rows)

def infer_current_max_generation(out_dir):
    p = load_progress(out_dir)
    if len(p) and "generation" in p.columns:
        vals = pd.to_numeric(p["generation"], errors="coerce").dropna()
        if len(vals):
            return int(vals.max())
    prompt_nums = []
    for pth in Path(out_dir).glob("advisor_prompt_generation_*.txt"):
        m = re.search(r"generation_(\d+)", pth.name)
        if m:
            prompt_nums.append(int(m.group(1)))
    return max(prompt_nums) if prompt_nums else 0

def summarize_run(out_dir, label="run"):
    out_dir = Path(out_dir)
    row = {
        "label": label,
        "out_dir": str(out_dir),
        "summary_exists": (out_dir / "ga_summary.json").exists(),
        "progress_exists": (out_dir / "ga_generation_progress.csv").exists(),
        "transitions_exists": (out_dir / "population_transitions.csv").exists(),
    }
    p = load_progress(out_dir)
    if len(p):
        for c in ["validation_det_pass_rate", "validation_avg_det_score", "best_so_far_DETPass"]:
            if c in p.columns:
                row[c] = float(pd.to_numeric(p[c], errors="coerce").fillna(0).max())
        if "avg_prompt_tokens" in p.columns:
            toks = pd.to_numeric(p["avg_prompt_tokens"], errors="coerce").dropna()
            if len(toks):
                row["first_tokens"] = float(toks.iloc[0])
                row["last_tokens"] = float(toks.iloc[-1])
                row["min_tokens"] = float(toks.min())
                row["token_delta"] = row["last_tokens"] - row["first_tokens"]
                row["best_token_reduction_ratio"] = (row["first_tokens"] - row["min_tokens"]) / row["first_tokens"] if row["first_tokens"] else np.nan
    t = load_transitions(out_dir)
    if len(t):
        for c in ["new_by_compression_fallback", "new_by_advisor", "advisor_proposals_generated", "advisor_proposals_accepted_applied", "advisor_proposals_rejected"]:
            if c in t.columns:
                row[c + "_sum"] = float(pd.to_numeric(t[c], errors="coerce").fillna(0).sum())
    prop = load_all_proposals(out_dir)
    row["proposal_rows"] = len(prop)
    row["fallback_rows"] = int((prop.get("source", pd.Series(dtype=str)).astype(str) == "compression_fallback").sum()) if len(prop) and "source" in prop.columns else 0
    return row

def display_run_summary(out_dir, label="run"):
    row = summarize_run(out_dir, label)
    display(pd.DataFrame([row]))
    return row

def latest_output_dir():
    if "cloud_only_by_config_out" in globals():
        return Path(cloud_only_by_config_out)
    raise RuntimeError("cloud_only_by_config_out is not defined. Run the initial cloud-only-by-config cell first.")


In [13]:
# ============================================================
# Cell 5. run_ga_all_categories wrapper
# - Uses only supported flags.
# - Cloud-only-by-config is achieved by quota/rate settings, not strict cloud-only flags.
# ============================================================

def flag_supported(flag: str) -> bool:
    return flag in SUPPORTED_FLAGS or flag in run_ga_src

def add_value(cmd, flag, value, *, required=False):
    if value is None:
        return
    if not flag_supported(flag):
        if required:
            raise RuntimeError(f"Required unsupported flag: {flag}")
        print("[SKIP unsupported flag]", flag)
        return
    cmd.extend([flag, str(value)])

def add_bool(cmd, flag, enabled, *, required=False):
    if not enabled:
        return
    if not flag_supported(flag):
        if required:
            raise RuntimeError(f"Required unsupported flag: {flag}")
        print("[SKIP unsupported flag]", flag)
        return
    cmd.append(flag)

def build_base_ga_cmd(
    *,
    output_root,
    model_key,
    categories,
    limit_per_category,
    sample_size,
    validation_size,
    population,
    gens,
    target_detpass=90,
    timeout_sec=2400,
    resume=False,
):
    cmd = [
        str(PYTHON), "-u", str(SCRIPT),
        "--profile", "version0_15",
        "--model-key", model_key,
        "--target-detpass", str(target_detpass),
        "--llm-mode", "worker",
        "--population", str(population),
        "--gens", str(gens),
        "--min-generations", str(gens),
        "--max-generations", str(gens),
        "--sample-size", str(sample_size),
        "--validation-size", str(validation_size),
        "--cheap-eval-limit", "2",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", "cloudless_decompiler",
        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--plateau-window", "1",
        "--disruptive-max-attempts", "1",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",
        "--progress", "verbose",
        "--timeout-sec", str(timeout_sec),
        "--retries", "0",
        "--limit-per-category", str(limit_per_category),
        "--output-root", str(output_root),
        "--feedback-guided-mutation",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",
        "--full-run",
    ]
    if resume:
        cmd.append("--resume")
    for c in categories:
        cmd += ["--category", str(c)]
    return cmd

def add_cloud_only_by_config_flags(cmd):
    # Local/fallback compression lanes are disabled by quota/rate.
    # Advisor compression child scheduling remains enabled.
    add_value(cmd, "--compression-child-quota", 0, required=True)
    add_value(cmd, "--compression-child-ratio", 0.0, required=True)
    add_value(cmd, "--micro-compression-child-quota", 0, required=True)
    add_value(cmd, "--micro-compression-child-ratio", 0.0, required=True)
    add_value(cmd, "--block-compression-child-quota", 0, required=True)
    add_value(cmd, "--block-compression-child-ratio", 0.0, required=True)
    add_value(cmd, "--multi-block-compression-child-quota", 0, required=True)
    add_value(cmd, "--multi-block-compression-child-ratio", 0.0, required=True)
    add_value(cmd, "--global-budget-compression-child-quota", 0, required=True)

    add_bool(cmd, "--llm-mutation-advisor", True, required=True)
    add_value(cmd, "--advisor-model-key", ADVISOR_MODEL_KEY, required=True)
    add_value(cmd, "--advisor-trigger-mode", "always", required=True)
    add_value(cmd, "--advisor-min-population-for-child", 4, required=True)
    add_bool(cmd, "--advisor-force-child-quota", True, required=True)
    add_value(cmd, "--advisor-compression-child-quota", 2, required=True)
    add_value(cmd, "--advisor-prefer-compression-after-detpass", 90, required=True)

    add_value(cmd, "--compression-detpass-threshold", 90, required=True)
    add_bool(cmd, "--aggressive-compression-after-target", True, required=True)
    add_value(cmd, "--compression-token-reduction-target", 0.15, required=True)
    add_value(cmd, "--compression-token-plateau-delta", 1.0, required=True)
    add_bool(cmd, "--allow-aggressive-compression", True, required=True)
    add_bool(cmd, "--enable-block-token-breakdown", True, required=True)
    add_bool(cmd, "--enable-multi-block-compression", True, required=True)
    add_value(cmd, "--min-compression-token-delta", 50, required=True)
    return cmd

def run_subprocess(cmd, timeout=7200):
    env = os.environ.copy()
    env["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
    env["JOI_V15_LOCAL_FILES_ONLY"] = "true"
    env["TRANSFORMERS_OFFLINE"] = env.get("TRANSFORMERS_OFFLINE", "1")
    env["HF_HUB_OFFLINE"] = env.get("HF_HUB_OFFLINE", "1")
    env["TOKENIZERS_PARALLELISM"] = "false"

    print("=" * 120)
    print("[COMMAND]")
    print("=" * 120)
    print(" ".join(shlex.quote(str(x)) for x in cmd))

    p = subprocess.run(
        list(map(str, cmd)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        timeout=timeout,
    )
    print(p.stdout)
    print("returncode:", p.returncode)
    if p.returncode != 0:
        raise RuntimeError(f"command failed rc={p.returncode}")
    return p

def run_initial_cloud_only_by_config():
    run_dir = RESULTS_ROOT / f"manual_cloud_only_by_config_{SERVER_PRESET}_cat{''.join(map(str, SMOKE_CATEGORIES))}_gens{INITIAL_GENS}_{SMOKE_MODEL_KEY}_{timestamp()}"
    out_dir = run_dir / "ga_output"
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = build_base_ga_cmd(
        output_root=out_dir,
        model_key=SMOKE_MODEL_KEY,
        categories=SMOKE_CATEGORIES,
        limit_per_category=SMOKE_LIMIT_PER_CATEGORY,
        sample_size=SMOKE_SAMPLE_SIZE,
        validation_size=SMOKE_VALIDATION_SIZE,
        population=SMOKE_POPULATION,
        gens=INITIAL_GENS,
        target_detpass=90,
        timeout_sec=2400,
        resume=False,
    )
    cmd = add_cloud_only_by_config_flags(cmd)
    run_subprocess(cmd, timeout=7200)
    return out_dir


In [14]:
# ============================================================
# Cell 6. Initial cloud-only-by-config run: generation 1..INITIAL_GENS
# Run this once before manual intervention.
# ============================================================

if RUN_INITIAL_SMOKE:
    cloud_only_by_config_out = run_initial_cloud_only_by_config()
    cloud_only_out = cloud_only_by_config_out  # alias
    print("cloud_only_by_config_out =", cloud_only_by_config_out)
    display_run_summary(cloud_only_by_config_out, "initial_cloud_only_by_config")
else:
    print("[SKIP] RUN_INITIAL_SMOKE=False")
    print("If you already have a run, set cloud_only_by_config_out = Path('/.../ga_output') manually.")


[COMMAND]
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_14b --target-detpass 90 --llm-mode worker --population 5 --gens 2 --min-generations 2 --max-generations 2 --sample-size 4 --validation-size 4 --cheap-eval-limit 2 --candidate-k 1 --repair-attempts 0 --det-profile strict --selection-mode redesign --fitness-mode phase_aware --mutation-mode cloudless_decompiler --category-balance-mode guard --token-penalty-mode hybrid --stop-controller-mode active --plateau-window 1 --disruptive-max-attempts 1 --reasoning-mutation-mode auto --intent-hint-mode auto --progress verbose --timeout-sec 2400 --retries 0 --limit-per-category 1 --output-root /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/manual_cloud_only_by_config_A100_SET_B_cat3456_gens2_qwen25_coder_14b_20260612_161553/ga_output --feedback-guided-mutation --enable-compression-mutation --enable-prompt-decompile

,label,out_dir,summary_exists,progress_exists,transitions_exists,validation_det_pass_rate,validation_avg_det_score,best_so_far_DETPass,first_tokens,last_tokens,min_tokens,token_delta,best_token_reduction_ratio,new_by_compression_fallback_sum,new_by_advisor_sum,advisor_proposals_generated_sum,advisor_proposals_accepted_applied_sum,advisor_proposals_rejected_sum,proposal_rows,fallback_rows
0,initial_cloud_only_by_config,/root/llm/JOILang-Server/gpt_mg/version0_15_up...,True,True,True,75.0,73.4821,75.0,39998.5,40024.5,39998.5,26.0,0.0,0.0,0.0,2.0,0.0,2.0,8,0


In [ ]:
# ============================================================
# Cell 7. Locate and inspect advisor prompt files
# advisor_prompt_generation_XXX.txt is a log, not the editable source.
# Editable source is advisor_feedback.py.
# ============================================================

OUT_DIR = latest_output_dir()

print("=" * 120)
print("[ADVISOR PROMPT FILES]")
print("=" * 120)
print("OUT_DIR:", OUT_DIR)

advisor_prompt_files = sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt"))
if not advisor_prompt_files:
    print("[WARN] No advisor_prompt_generation_*.txt files found.")
else:
    rows = []
    for p in advisor_prompt_files:
        m = re.search(r"generation_(\d+)", p.name)
        rows.append({
            "generation": int(m.group(1)) if m else None,
            "file": p.name,
            "path": str(p),
            "size_bytes": p.stat().st_size,
            "chars": len(p.read_text(encoding="utf-8", errors="replace")),
        })
    prompt_file_df = pd.DataFrame(rows)
    display(prompt_file_df)

    latest = advisor_prompt_files[-1]
    print("\n[LATEST PROMPT PREVIEW]", latest)
    text = latest.read_text(encoding="utf-8", errors="replace")
    print(text[:6000])

print("\n[EDITABLE SOURCE]")
print("ADVISOR_FEEDBACK_PATH:", ADVISOR_FEEDBACK_PATH)


## Manual intervention protocol

이 셀 다음부터는 사용자가 직접 개입한다.

1. 위에서 `advisor_prompt_generation_005.txt`를 확인한다.
2. 아래 Cell 8로 `advisor_feedback.py`를 백업하고 현재 상태를 기록한다.
3. VS Code 또는 터미널에서 `ADVISOR_FEEDBACK_PATH`를 수정한다.
4. Cell 9로 수정 diff와 문법을 확인한다.
5. Cell 10에서 generation 6만 resume한다.
6. Cell 11~16으로 generation 5 vs 6의 advisor prompt, proposal, block, token, DETPass 변화를 확인한다.
7. 다시 수정하고 `NEXT_GEN=7` 또는 자동 추론으로 generation 7을 resume한다.


In [ ]:
# ============================================================
# Cell 8. Snapshot advisor_feedback.py before manual edit
# This creates a timestamped backup and records the baseline hash.
# ============================================================

OUT_DIR = latest_output_dir()
current_gen = infer_current_max_generation(OUT_DIR)

snapshot_id = f"before_edit_after_gen_{current_gen}_{timestamp()}"
backup_path = PATCH_DIR / f"advisor_feedback_{snapshot_id}.py"
shutil.copy2(ADVISOR_FEEDBACK_PATH, backup_path)

advisor_feedback_baseline = {
    "snapshot_id": snapshot_id,
    "out_dir": str(OUT_DIR),
    "current_generation": current_gen,
    "source_path": str(ADVISOR_FEEDBACK_PATH),
    "backup_path": str(backup_path),
    "sha256": sha256_file(ADVISOR_FEEDBACK_PATH),
    "created_at": datetime.now().isoformat(timespec="seconds"),
}

baseline_json = PATCH_DIR / f"advisor_feedback_{snapshot_id}.json"
baseline_json.write_text(json.dumps(advisor_feedback_baseline, indent=2, ensure_ascii=False), encoding="utf-8")

print("=" * 120)
print("[ADVISOR_FEEDBACK SNAPSHOT]")
print("=" * 120)
print(json.dumps(advisor_feedback_baseline, indent=2, ensure_ascii=False))
print("\nNow edit this file:")
print(ADVISOR_FEEDBACK_PATH)


In [ ]:
# ============================================================
# Cell 9. Check advisor_feedback.py after manual edit
# - Shows git diff for the file.
# - Runs py_compile to catch syntax errors before a long GA resume.
# ============================================================

print("=" * 120)
print("[GIT DIFF: advisor_feedback.py]")
print("=" * 120)
run_cmd(["git", "-C", str(REPO), "diff", "--", str(ADVISOR_FEEDBACK_PATH.relative_to(REPO))], timeout=60, check=False)

print("=" * 120)
print("[PY_COMPILE]")
print("=" * 120)
run_cmd([PYTHON, "-m", "py_compile", str(ADVISOR_FEEDBACK_PATH)], timeout=120, check=True)

advisor_feedback_after_edit = {
    "source_path": str(ADVISOR_FEEDBACK_PATH),
    "sha256": sha256_file(ADVISOR_FEEDBACK_PATH),
    "checked_at": datetime.now().isoformat(timespec="seconds"),
}
print(json.dumps(advisor_feedback_after_edit, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# Cell 10. Resume one generation after advisor_feedback.py edit
# Set RUN_RESUME_ONE_GEN=True after editing and compiling advisor_feedback.py.
# NEXT_GEN=None means infer current max generation + 1.
# ============================================================

def resume_cloud_only_by_config_one_generation(out_dir, next_gen=None):
    out_dir = Path(out_dir)
    current = infer_current_max_generation(out_dir)
    gen = int(next_gen or (current + 1))

    cmd = build_base_ga_cmd(
        output_root=out_dir,
        model_key=SMOKE_MODEL_KEY,
        categories=SMOKE_CATEGORIES,
        limit_per_category=SMOKE_LIMIT_PER_CATEGORY,
        sample_size=SMOKE_SAMPLE_SIZE,
        validation_size=SMOKE_VALIDATION_SIZE,
        population=SMOKE_POPULATION,
        gens=gen,
        target_detpass=90,
        timeout_sec=2400,
        resume=True,
    )
    cmd = add_cloud_only_by_config_flags(cmd)

    print("=" * 120)
    print("[RESUME ONE GENERATION]")
    print("=" * 120)
    print("OUT_DIR:", out_dir)
    print("current max generation:", current)
    print("next generation:", gen)

    run_subprocess(cmd, timeout=7200)
    return gen

OUT_DIR = latest_output_dir()

if RUN_RESUME_ONE_GEN:
    resumed_generation = resume_cloud_only_by_config_one_generation(OUT_DIR, NEXT_GEN)
    resume_marker = {
        "out_dir": str(OUT_DIR),
        "resumed_generation": resumed_generation,
        "advisor_feedback_sha256": sha256_file(ADVISOR_FEEDBACK_PATH),
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    marker_path = PATCH_DIR / f"resume_generation_{resumed_generation}_{timestamp()}.json"
    marker_path.write_text(json.dumps(resume_marker, indent=2, ensure_ascii=False), encoding="utf-8")
    print("resume marker saved:", marker_path)
else:
    print("[SKIP] RUN_RESUME_ONE_GEN=False")
    print("After editing advisor_feedback.py and running Cell 9, set RUN_RESUME_ONE_GEN=True and rerun this cell.")


In [ ]:
# ============================================================
# Cell 11. Compare advisor prompt before/after resume
# Auto compares last two advisor_prompt_generation_*.txt files.
# ============================================================

OUT_DIR = latest_output_dir()
prompt_files = sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt"))

if len(prompt_files) < 2:
    print("[WARN] Need at least two advisor prompt files.")
else:
    before = prompt_files[-2]
    after = prompt_files[-1]

    before_lines = before.read_text(encoding="utf-8", errors="replace").splitlines()
    after_lines = after.read_text(encoding="utf-8", errors="replace").splitlines()

    print("=" * 120)
    print("[ADVISOR PROMPT BEFORE/AFTER]")
    print("=" * 120)
    print("BEFORE:", before)
    print("AFTER :", after)
    print("before chars:", sum(len(x) for x in before_lines))
    print("after chars :", sum(len(x) for x in after_lines))
    print("delta chars :", sum(len(x) for x in after_lines) - sum(len(x) for x in before_lines))

    diff = difflib.unified_diff(
        before_lines,
        after_lines,
        fromfile=before.name,
        tofile=after.name,
        lineterm="",
        n=5,
    )
    diff_text = "\n".join(list(diff))
    print(diff_text[:30000])

    diff_path = TABLE_DIR / f"advisor_prompt_diff_{before.stem}_vs_{after.stem}_{timestamp()}.diff"
    diff_path.write_text(diff_text, encoding="utf-8")
    print("\nsaved diff:", diff_path)

    html_path = TABLE_DIR / f"advisor_prompt_diff_{before.stem}_vs_{after.stem}_{timestamp()}.html"
    html = difflib.HtmlDiff(wrapcolumn=120).make_file(before_lines, after_lines, before.name, after.name)
    html_path.write_text(html, encoding="utf-8")
    print("saved html diff:", html_path)


In [ ]:
# ============================================================
# Cell 12. Advisor response/proposal table after manual intervention
# ============================================================

OUT_DIR = latest_output_dir()

resp_rows = []
for p in sorted(OUT_DIR.glob("advisor_response_generation_*.json")):
    obj = read_json(p) or {}
    parsed = obj.get("parsed", {})
    raw = str(obj.get("raw_content", ""))
    m = re.search(r"generation_(\d+)", p.name)
    gen = int(m.group(1)) if m else None
    resp_rows.append({
        "generation": gen,
        "file": p.name,
        "size_bytes": p.stat().st_size,
        "parsed_type": type(parsed).__name__,
        "parsed_keys": list(parsed.keys()) if isinstance(parsed, dict) else [],
        "accepted_proposals": len(obj.get("accepted_proposals", []) or []),
        "rejected_proposals": len(obj.get("rejected_proposals", []) or []),
        "raw_len": len(raw),
        "raw_head": raw[:300],
    })

resp_df = pd.DataFrame(resp_rows)
display(resp_df.tail(10))

proposal_df = load_all_proposals(OUT_DIR)
if len(proposal_df):
    cols = [c for c in [
        "_file", "generation", "source", "schema_source", "case_label", "case_label_inferred",
        "operator", "mutation_type", "compression_level", "target_block_id", "selected_block_id",
        "selected_block_ids", "expected_token_delta", "measured_prompt_token_delta",
        "accepted", "applied", "rejection_reason", "fallback_reason"
    ] if c in proposal_df.columns]
    display(proposal_df[cols].tail(50))

    if "case_label_inferred" in proposal_df.columns:
        display(proposal_df.groupby(["case_label_inferred"], dropna=False).size().reset_index(name="count"))
else:
    print("[WARN] No proposal rows found.")


In [ ]:
# ============================================================
# Cell 13. Progress and DETPass/token trend
# Generates readable tables and standalone figures.
# ============================================================

OUT_DIR = latest_output_dir()
progress = load_progress(OUT_DIR)
trans = load_transitions(OUT_DIR)

print("=" * 120)
print("[PROGRESS]")
print("=" * 120)
if len(progress):
    cols = [c for c in [
        "generation", "validation_det_pass_rate", "validation_avg_det_score", "best_so_far_DETPass",
        "avg_prompt_tokens", "compression_ready", "compression_phase", "advisor_triggered",
    ] if c in progress.columns]
    display(progress[cols].tail(20))
else:
    print("[WARN] progress empty")

print("=" * 120)
print("[TRANSITIONS]")
print("=" * 120)
if len(trans):
    cols = [c for c in [
        "generation", "new_by_advisor", "advisor_proposals_generated",
        "advisor_proposals_accepted_applied", "advisor_proposals_rejected",
        "new_by_compression_fallback", "new_by_compression",
        "compression_ready", "compression_phase",
    ] if c in trans.columns]
    display(trans[cols].tail(20))
else:
    print("[WARN] transitions empty")

try:
    import matplotlib.pyplot as plt

    if len(progress) and "generation" in progress.columns and "avg_prompt_tokens" in progress.columns:
        plt.figure()
        plt.plot(progress["generation"], pd.to_numeric(progress["avg_prompt_tokens"], errors="coerce"), marker="o")
        plt.xlabel("Generation")
        plt.ylabel("Average prompt tokens")
        plt.title("Prompt token trend by generation")
        plt.grid(True)
        fig_path = FIGURE_DIR / f"avg_prompt_tokens_{timestamp()}.png"
        plt.savefig(fig_path, bbox_inches="tight", dpi=150)
        plt.show()
        print("saved:", fig_path)

    if len(progress) and "generation" in progress.columns:
        det_cols = [c for c in ["validation_det_pass_rate", "best_so_far_DETPass"] if c in progress.columns]
        if det_cols:
            plt.figure()
            for c in det_cols:
                plt.plot(progress["generation"], pd.to_numeric(progress[c], errors="coerce"), marker="o", label=c)
            plt.xlabel("Generation")
            plt.ylabel("DETPass / pass rate")
            plt.title("DETPass trend by generation")
            plt.legend()
            plt.grid(True)
            fig_path = FIGURE_DIR / f"detpass_trend_{timestamp()}.png"
            plt.savefig(fig_path, bbox_inches="tight", dpi=150)
            plt.show()
            print("saved:", fig_path)

    if len(trans) and "generation" in trans.columns:
        count_cols = [c for c in ["new_by_advisor", "advisor_proposals_generated", "advisor_proposals_accepted_applied", "advisor_proposals_rejected", "new_by_compression_fallback"] if c in trans.columns]
        if count_cols:
            plot_df = trans[["generation"] + count_cols].copy()
            for c in count_cols:
                plot_df[c] = pd.to_numeric(plot_df[c], errors="coerce").fillna(0)
            plt.figure()
            for c in count_cols:
                plt.plot(plot_df["generation"], plot_df[c], marker="o", label=c)
            plt.xlabel("Generation")
            plt.ylabel("Count")
            plt.title("Advisor/compression transition counts")
            plt.legend()
            plt.grid(True)
            fig_path = FIGURE_DIR / f"transition_counts_{timestamp()}.png"
            plt.savefig(fig_path, bbox_inches="tight", dpi=150)
            plt.show()
            print("saved:", fig_path)

except Exception as e:
    print("[WARN] plotting failed:", repr(e))


In [ ]:
# ============================================================
# Cell 14. Block/genome diff inspection
# Shows how JOI prompt blocks changed across generations.
# ============================================================

OUT_DIR = latest_output_dir()
diff_path = OUT_DIR / "ga_block_diffs.jsonl"

rows = read_jsonl(diff_path)
print("diff_path:", diff_path, "rows:", len(rows))

flat_rows = []
for obj in rows:
    if not isinstance(obj, dict):
        continue
    base = {k: obj.get(k) for k in ["generation", "genome_id", "parent_id", "child_id", "proposal_id"]}
    found = False
    for key in ["changed", "changes", "diffs", "block_diffs", "mutations"]:
        vals = obj.get(key)
        if isinstance(vals, list):
            for item in vals:
                rr = dict(base)
                if isinstance(item, dict):
                    rr.update(item)
                else:
                    rr["change"] = str(item)
                flat_rows.append(rr)
                found = True
    if not found:
        flat_rows.append(obj)

block_diff_df = pd.DataFrame(flat_rows)
if len(block_diff_df):
    display(block_diff_df.tail(100))
    if "generation" in block_diff_df.columns:
        current = infer_current_max_generation(OUT_DIR)
        display(block_diff_df[pd.to_numeric(block_diff_df["generation"], errors="coerce").isin([current-1, current])])
else:
    print("[WARN] No block diff rows found.")

out_csv = TABLE_DIR / f"block_diff_flat_{timestamp()}.csv"
block_diff_df.to_csv(out_csv, index=False)
print("saved:", out_csv)


In [ ]:
# ============================================================
# Cell 15. Actual JOI code generation prompt logs
# Finds prompt_log_paths from candidate CSVs and compares recent prompts.
# ============================================================

OUT_DIR = latest_output_dir()
cand_dir = OUT_DIR / "candidates"

prompt_log_rows = []
if cand_dir.exists():
    for csv_path in sorted(cand_dir.glob("*.csv")):
        try:
            df = pd.read_csv(csv_path)
        except Exception:
            continue
        if "prompt_log_paths" not in df.columns:
            continue
        for _, row in df.iterrows():
            raw = row.get("prompt_log_paths")
            if pd.isna(raw):
                continue
            try:
                parsed = ast.literal_eval(raw)
                paths = parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                paths = [raw]
            for p in paths:
                p = str(p).strip()
                if not p:
                    continue
                pp = Path(p)
                prompt_log_rows.append({
                    "candidate_csv": csv_path.name,
                    "prompt_log_path": str(pp),
                    "exists": pp.exists(),
                    "size": pp.stat().st_size if pp.exists() else None,
                })

prompt_log_df = pd.DataFrame(prompt_log_rows)
display(prompt_log_df.tail(100))
print("prompt log count:", len(prompt_log_df))

valid_logs = prompt_log_df[prompt_log_df["exists"] == True]["prompt_log_path"].tolist() if len(prompt_log_df) else []

if len(valid_logs) >= 2:
    before_path = Path(valid_logs[-2])
    after_path = Path(valid_logs[-1])

    before_lines = before_path.read_text(encoding="utf-8", errors="replace").splitlines()
    after_lines = after_path.read_text(encoding="utf-8", errors="replace").splitlines()

    print("=" * 120)
    print("[ACTUAL JOI GENERATION PROMPT DIFF]")
    print("=" * 120)
    print("BEFORE:", before_path)
    print("AFTER :", after_path)
    print("before chars:", sum(len(x) for x in before_lines))
    print("after chars :", sum(len(x) for x in after_lines))
    print("delta chars :", sum(len(x) for x in after_lines) - sum(len(x) for x in before_lines))

    diff_text = "\n".join(difflib.unified_diff(
        before_lines,
        after_lines,
        fromfile=str(before_path),
        tofile=str(after_path),
        lineterm="",
        n=5,
    ))
    print(diff_text[:30000])

    diff_path = TABLE_DIR / f"actual_joi_prompt_diff_{timestamp()}.diff"
    diff_path.write_text(diff_text, encoding="utf-8")
    print("saved:", diff_path)
else:
    print("[WARN] Need at least two actual prompt log files.")


In [ ]:
# ============================================================
# Cell 16. Cloud-only-by-config final gate
# Fallback must be zero. Strict cloud-only artifacts are optional.
# ============================================================

OUT_DIR = latest_output_dir()
errors, warnings = [], []

progress = load_progress(OUT_DIR)
trans = load_transitions(OUT_DIR)
prop = load_all_proposals(OUT_DIR)

for fn in ["ga_summary.json", "ga_generation_progress.csv", "population_transitions.csv"]:
    if not (OUT_DIR / fn).exists():
        errors.append(f"missing {fn}")

if len(progress):
    if "avg_prompt_tokens" not in progress.columns or pd.to_numeric(progress["avg_prompt_tokens"], errors="coerce").fillna(0).max() <= 0:
        errors.append("avg_prompt_tokens missing or non-positive")
    det_candidates = []
    for c in ["best_so_far_DETPass", "validation_det_pass_rate", "det_pass_rate"]:
        if c in progress.columns:
            det_candidates.append(float(pd.to_numeric(progress[c], errors="coerce").fillna(0).max()))
    det_max = max(det_candidates) if det_candidates else 0
    if det_max < 90:
        errors.append(f"DETPass below 90: {det_max}")
else:
    errors.append("progress empty")

fallback_rows = 0
if len(prop) and "source" in prop.columns:
    fallback_rows = int((prop["source"].astype(str) == "compression_fallback").sum())
    if fallback_rows:
        errors.append(f'source="compression_fallback" rows exist: {fallback_rows}')

fallback_sum = 0
if len(trans) and "new_by_compression_fallback" in trans.columns:
    fallback_sum = float(pd.to_numeric(trans["new_by_compression_fallback"], errors="coerce").fillna(0).sum())
    if fallback_sum != 0:
        errors.append(f"new_by_compression_fallback_sum={fallback_sum}")
else:
    warnings.append("new_by_compression_fallback column not found; fallback row check is used instead.")

advisor_rows = int((prop.get("_file", pd.Series(dtype=str)) == "advisor_mutation_proposals.jsonl").sum()) if len(prop) else 0
if advisor_rows <= 0:
    warnings.append("advisor proposal rows == 0. Cloud-only isolation can still be checked, but advisor contribution is not observable.")

accepted = int((prop.get("accepted", pd.Series(dtype=bool)) == True).sum()) if len(prop) and "accepted" in prop.columns else 0
applied = int((prop.get("applied", pd.Series(dtype=bool)) == True).sum()) if len(prop) and "applied" in prop.columns else 0
if accepted + applied <= 0:
    warnings.append("advisor accepted/applied count is 0. Advisor proposal quality may be weak.")

for optional_artifact in ["advisor_cloud_only_before_after.csv", "advisor_case_ABC_summary.csv"]:
    if not (OUT_DIR / optional_artifact).exists():
        warnings.append(f"{optional_artifact} not found. Expected without strict cloud-only implementation.")

print("=" * 120)
print("[FINAL GATE: CLOUD-ONLY-BY-CONFIG]")
print("=" * 120)
print("OUT_DIR:", OUT_DIR)
print("fallback_rows:", fallback_rows)
print("new_by_compression_fallback_sum:", fallback_sum)
print("advisor_rows:", advisor_rows)
print("advisor_accepted:", accepted)
print("advisor_applied:", applied)

if warnings:
    print("\n[WARNINGS]")
    for w in warnings:
        print("-", w)

if errors:
    print("\n[DO NOT PROMOTE]")
    for e in errors:
        print("-", e)
    raise RuntimeError("cloud-only-by-config gate failed")

print("\n[OK] cloud-only-by-config gate passed")


In [ ]:
# ============================================================
# Cell 17. Save experiment ledger and paper-ready summaries
# ============================================================

OUT_DIR = latest_output_dir()
current_gen = infer_current_max_generation(OUT_DIR)

summary = summarize_run(OUT_DIR, f"cloud_only_by_config_until_gen_{current_gen}")
summary["advisor_feedback_sha256"] = sha256_file(ADVISOR_FEEDBACK_PATH)
summary["current_generation"] = current_gen
summary["saved_at"] = datetime.now().isoformat(timespec="seconds")

summary_df = pd.DataFrame([summary])
display(summary_df)

summary_csv = SUMMARY_DIR / f"manual_intervention_summary_gen{current_gen}_{timestamp()}.csv"
summary_df.to_csv(summary_csv, index=False)
print("saved summary:", summary_csv)

# Save compact markdown report.
report_lines = []
report_lines.append(f"# JOILang cloud-only-by-config manual intervention report\n")
report_lines.append(f"- out_dir: `{OUT_DIR}`")
report_lines.append(f"- current_generation: {current_gen}")
report_lines.append(f"- advisor_feedback.py: `{ADVISOR_FEEDBACK_PATH}`")
report_lines.append(f"- advisor_feedback_sha256: `{sha256_file(ADVISOR_FEEDBACK_PATH)}`")
report_lines.append("")
report_lines.append("## Summary")
for k, v in summary.items():
    report_lines.append(f"- {k}: {v}")

report_path = SUMMARY_DIR / f"manual_intervention_report_gen{current_gen}_{timestamp()}.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print("saved report:", report_path)


## Recommended manual loop

반복 실험은 다음 순서로 진행한다.

```text
A. Cell 6: generation 1~5 초기 실행
B. Cell 7: advisor_prompt_generation_005.txt 확인
C. Cell 8: advisor_feedback.py 백업
D. 사용자가 advisor_feedback.py 직접 수정
E. Cell 9: git diff + py_compile
F. Cell 10: RUN_RESUME_ONE_GEN=True로 generation 6 실행
G. Cell 11~16: 5→6 변화 분석
H. 다시 advisor_feedback.py 수정
I. Cell 10: NEXT_GEN=None 상태로 다시 실행하면 generation 7 자동 실행
J. Cell 11~17 재실행
```

주의:
- `advisor_prompt_generation_XXX.txt`는 로그 파일이다. 직접 수정해도 다음 generation에 반영되지 않는다.
- 수정 대상은 `scripts/advisor_feedback.py`이다.
- `py_compile`은 성능 평가가 아니라 긴 실행 전에 문법 오류를 빠르게 잡는 안전 점검이다.
- `source="compression_fallback"`가 한 줄이라도 생기면 해당 run은 cloud-only-by-config라고 주장하면 안 된다.
